In [0]:
class Silver_qualifying():
    main_path="/Volumes/formula1_race/default/formula1/"
    bronze_path = "formula1_race_project/bronze"
    silver_path = "formula1_race_project/silver"

    def __init__(self,folder):
         self.folder_name=folder
        
    def read_input(self):
        from pyspark.sql.functions import col,max,count
        if spark.catalog.tableExists("formula1_race.silver.qualifying"):
            max_ingestion_date=spark.read.table('formula1_race.silver.qualifying').agg(max(col('qualifying_ingestion_date')).alias('max_ingestion_date')).collect()[0]['max_ingestion_date']
            if max_ingestion_date is None:
                max_ingestion_date='1900-01-01 00:00:00'
            print("if_max_ingestion_date:",max_ingestion_date)
            print(f"if_max_ingestion_date:",type(max_ingestion_date))
            read_df=spark.read.table('formula1_race.bronze.qualifying').filter(col('QualifyingIngestionDate')>max_ingestion_date)
    
            print("reading qualifying read_df")
            display(read_df.select(count("*")))
        else:
            max_ingestion_date='1900-01-01 00:00:00'
            print("else_max_ingestion_date:",max_ingestion_date)
            read_df=spark.read.table('formula1_race.bronze.qualifying').filter(col('QualifyingIngestionDate')>max_ingestion_date)
        return read_df
    
    def column_name_formating(self,read_df):
        colrename_df=read_df
        import re
        for c in colrename_df.columns:
            result = re.sub(r'([a-z])([A-Z])',r'\1,\2',c).lower().split(',')
            new_column="_".join(result)
            colrename_df=colrename_df.withColumnRenamed(c,new_column)
        return colrename_df
    
    def apply_transformations(self,colrename_df):
        from pyspark.sql.functions import round,col,when
        apply_tran_df=colrename_df
        for c,t in apply_tran_df.dtypes:
            if t=='string':
                print(c)
                apply_tran_df=apply_tran_df.withColumn(c,when(col(c).isin('\\N',''),'-').otherwise(col(c)))

        apply_tran_df= (apply_tran_df.select(col('race_id'),col('driver_id'),col('constructor_id'),
                                             col('number'),col('position'),col('q1'),col('q2'),col('q3'),
                                             col('qualifying_ingestion_date'),col('source'))
                        )
        return apply_tran_df
    
    def write_output(self,apply_tran_df):
        apply_tran_df.write.partitionBy("race_id").mode("append").saveAsTable("formula1_race.silver.qualifying")
        display(spark.sql(f"select count(*) from formula1_race.silver.qualifying"))
        print("Data write into sliver qualifying table is Done")
    
     
    def process(self):
        print("Started silver-ingestion-qualifying  in ran....")
        read_df=self.read_input()
        colrename_df=self.column_name_formating(read_df)
        apply_tran_df=self.apply_transformations(colrename_df)
        self.write_output(apply_tran_df)

In [0]:
Silver_qualifying_instance = Silver_qualifying("qualifying")
Silver_qualifying_instance.process()